In [ ]:
# Import necessary modules
import os
import pandas as pd
import datetime
import matplotlib.pyplot as plt
import numpy as np
import json
import requests
from IPython.display import clear_output, display
import time
import sqlite3
import random
from datetime import datetime, timedelta

logged_in_user_id = None

def log_access(event_description, user_id=None, log_type="General"):
    """
    Logs an access event into the AccessLogs table.

    Parameters:
        event_description (str): A description of the event being logged.
        user_id (int, optional): The ID of the user associated with the event. Defaults to None.
        log_type (str): The type of log event (e.g., "General", "Login", "Update"). Defaults to "General".
    """
    db_path = os.path.join(os.getcwd(), "VegasIQ.db")
    if not os.path.exists(db_path):
        print("Database not found. Please initialize the system first.")
        return

    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        # Ensure the LogType exists in LogTypeDimension
        cursor.execute("""
            INSERT OR IGNORE INTO LogTypeDimension (LogType, LogTypeDescription, FirstSeenDate, LastSeenDate, errorCount)
            VALUES (?, ?, DATE('now'), DATE('now'), 0)
        """, (log_type, f"{log_type} event logged."))

        # Update the LastSeenDate for the log type
        cursor.execute("""
            UPDATE LogTypeDimension
            SET LastSeenDate = DATE('now')
            WHERE LogType = ?
        """, (log_type,))

        # Retrieve the LogTypeID
        cursor.execute("""
            SELECT LogTypeID FROM LogTypeDimension WHERE LogType = ?
        """, (log_type,))
        log_type_id = cursor.fetchone()[0]

        # Insert the log into AccessLogs
        cursor.execute("""
            INSERT INTO AccessLogs (UserID, AccessDateTime, LogTypeID, create_date, update_date)
            VALUES (?, DATETIME('now'), ?, DATE('now'), DATE('now'))
        """, (user_id, log_type_id))

        conn.commit()
        

    except sqlite3.Error as e:
        print(f"An error occurred while logging access: {e}")

    finally:
        if 'conn' in locals() and conn:
            conn.close()

def authenticate_user():
    """
    Displays the authentication splash screen and validates the user's credentials.
    Sets the global `logged_in_user_id` if authentication is successful.
    Returns the permission level of the user if authentication is successful, otherwise None.
    """
    global logged_in_user_id
    db_path = os.path.join(os.getcwd(), "VegasIQ.db")
    
    if not os.path.exists(db_path):
        print("Database not found. Please initialize the system first.")
        return None

    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        print("Welcome to VegasIQ - User Authentication")
        print("========================================")

        while True:
            username = input("Enter your username: ").strip()
            if not username:
                print("Username cannot be empty. Please try again.")
                continue

            password = input("Enter your password: ").strip()
            if not password:
                print("Password cannot be empty. Please try again.")
                continue
            break
        
        # Query the database to check for matching credentials and get the permission level
        cursor.execute("""
            SELECT UserID, permission_level 
            FROM UserTable 
            WHERE username = ? AND password = ?
        """, (username, password))

        user = cursor.fetchone()

        if user:
            logged_in_user_id = user[0]  # Store the UserID in the global variable
            permission_level = user[1]
            log_access(event_description=f"User {username} has logged in to the system", log_type="System Log in", user_id=logged_in_user_id)
            print("Authentication successful!")
            input("Press Enter to continue to the main menu...")
            return permission_level
        else:
            print("Invalid username or password. Please try again.")
            input("Press Enter to retry...")
            return None

    except sqlite3.Error as e:
        print(f"An error occurred while accessing the database: {e}")
        return None

    finally:
        if 'conn' in locals() and conn:
            conn.close()




# Function to initialize the database
def initialize_database_once():
    db_path = os.path.join(os.getcwd(), "VegasIQ.db")
    if not os.path.exists(db_path):
        try:
            conn = sqlite3.connect(db_path)
            cursor = conn.cursor()

            # Create the UserTable
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS UserTable (
                    UserID INTEGER PRIMARY KEY AUTOINCREMENT,
                    first_name TEXT NOT NULL,
                    last_name TEXT NOT NULL,
                    username TEXT NOT NULL,
                    password TEXT NOT NULL,
                    permission_level INTEGER NOT NULL CHECK(permission_level IN (1, 2)),
                    create_date DATE NOT NULL,
                    update_date DATE NOT NULL
                )
            """)

            # Create the AccessLogs table
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS AccessLogs (
                    UserID INTEGER NULL,
                    AccessDateTime DATETIME NOT NULL,
                    LogTypeID INTEGER NOT NULL,
                    create_date DATE NOT NULL,
                    update_date DATE NOT NULL,
                    FOREIGN KEY(UserID) REFERENCES UserTable(UserID),
                    FOREIGN KEY(LogTypeID) REFERENCES LogTypeDimension(LogTypeID)
                )
            """)

            # Create the LogTypeDimension table
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS LogTypeDimension (
                    LogTypeID INTEGER PRIMARY KEY AUTOINCREMENT,
                    LogType TEXT NOT NULL,
                    LogTypeDescription TEXT NOT NULL,
                    FirstSeenDate DATE NOT NULL,
                    LastSeenDate DATE NOT NULL,
                    errorCount INTEGER NOT NULL
                )
            """)

            # Populate the UserTable with 200 rows of sample data
            first_names = ["Jorge", "Keith", "Bereniz", "Ola", "Quan"]
            last_names = ["Alvarez", "Webster", "Castaneda", "Nguyen", "Balogun"]
            now = datetime.now()

            for _ in range(200):
                first_name = random.choice(first_names)
                last_name = random.choice(last_names)
                username = f"{first_name.lower()}.{last_name.lower()}{random.randint(1, 100)}"
                password = "DefaultPassword"
                permission_level = random.choice([1, 2])
                create_date = (now - timedelta(days=random.randint(0, 365))).strftime("%Y-%m-%d")
                update_date = create_date

                cursor.execute("""
                    INSERT INTO UserTable (first_name, last_name, username, password, permission_level, create_date, update_date)
                    VALUES (?, ?, ?, ?, ?, ?, ?)
                """, (first_name, last_name, username, password, permission_level, create_date, update_date))

            # Populate the UserTable with predefined users
            predefined_users = [
                ("Root", "Admin", "root", "admin", 2),  # Administrator user
                ("Regular", "User", "user", "password", 1)  # Regular user
            ]

            now = datetime.now().strftime("%Y-%m-%d")
            for first_name, last_name, username, password, permission_level in predefined_users:
                cursor.execute("""
                    INSERT INTO UserTable (first_name, last_name, username, password, permission_level, create_date, update_date)
                    VALUES (?, ?, ?, ?, ?, ?, ?)
                """, (first_name, last_name, username, password, permission_level, now, now))


            # commit to database
            conn.commit()
            print(f"Database initialized and populated successfully at {db_path}.")

        except sqlite3.Error as e:
            print(f"An error occurred while interacting with the database: {e}")

        except Exception as ex:
            print(f"An unexpected error occurred: {ex}")

        finally:
            if 'conn' in locals() and conn:
                conn.close()
    else:
        print("Exisiting Databse found. Skipping initialization.")


# Main menu display function
def display_main_menu(permission_level):
    """
    Displays the main menu dynamically based on the user's permission level.
    """
    print("Welcome to VegasIQ")
    print("===================")
    print("1. Las Vegas Events and Rooms Database")
    print("2. Las Vegas Weather Data")
    print("3. Las Vegas Airport Statistics")
    print("4. Clark County Meeting Space Inventory")
    if permission_level == 2:  # Show "User Management" only for level 2 users
        print("5. User Management")
    print("6. System Exit")
    print("===================")
    try:
        return int(input("Select an option: "))
    except ValueError:
        return -1  # Return an invalid option to trigger error handling   return -1  # Return an invalid option to trigger error handlin


def user_management_menu():
    """
    Displays the user management menu and allows the admin to choose an action.
    """
    while True:
        clear_output(wait=True)
        print("User Management Menu")
        print("====================")
        print("1. Create a New User")
        print("2. Delete a User")
        print("3. Search for Users by Partial Username")
        print("4. Update Password by Partial Username Search")
        print("5. Update Permission Level by Partial Username Search")
        print("6. Return to Main Menu")
        print("====================")
        try:
            display("")
            return int(input("Select an option: "))
        except ValueError:
            print("Invalid input. Please try again.")
            continue


def search_users_by_partial_username(cursor, partial_username):
    """
    Searches for users by a partial username and handles cases with more than 10 results.
    Returns a list of users to the caller.
    """
    while True:
        cursor.execute("""
            SELECT UserID, username, first_name, last_name, permission_level
            FROM UserTable
            WHERE username LIKE ?
        """, (f"%{partial_username}%",))

        results = cursor.fetchall()

        if len(results) > 10:
            print(f"More than 10 users found ({len(results)}). Please refine your search or type 'exit' to go back.")
            partial_username = input("Enter a more specific search string: ").strip()
            if partial_username.lower() == 'exit':
                print("Exiting search.")
                return []
        elif len(results) == 0:
            print("No users found. Please try again or type 'exit' to go back.")
            partial_username = input("Enter a search string: ").strip()
            if partial_username.lower() == 'exit':
                print("Exiting search.")
                return []
        else:
            print("\nUsers Found:")
            for idx, (user_id, username, first_name, last_name, permission_level) in enumerate(results, 1):
                print(f"{idx}. UserID: {user_id}, Username: {username}, Name: {first_name} {last_name}, Permission Level: {permission_level}")
            return results



def create_user(cursor):
    """
    Creates a new user in the database.
    """
    while True:
        first_name = input("Enter first name: ").strip()
        if not first_name.isalpha():
            print("First name should contain only letters. Please try again.")
            continue
        break

    while True:
        last_name = input("Enter last name: ").strip()
        if not last_name.isalpha():
            print("Last name should contain only letters. Please try again.")
            continue
        break

    username = input("Enter username: ").strip()
    
    while True:
        password = input("Enter password: ").strip()
        if (len(password) < 8 or not any(char.isdigit() for char in password) or
                not any(char.isupper() for char in password) or not any(char.islower() for char in password) or
                not any(char in "!@#$%^&*()-_+=<>?" for char in password)):
            print("Password must be at least 8 characters long and include an uppercase letter, a lowercase letter, a number, and a special character. Please try again.")
            continue
        break

    while True:
        permission_level = input("Enter permission level (1 for regular user, 2 for admin): ").strip()
        if permission_level not in ('1', '2'):
            print("Permission level must be either 1 or 2. Please try again.")
            continue
        break

    try:
        cursor.execute("""
            INSERT INTO UserTable (first_name, last_name, username, password, permission_level, create_date, update_date)
            VALUES (?, ?, ?, ?, ?, DATE('now'), DATE('now'))
        """, (first_name, last_name, username, password, int(permission_level)))
        print("User created successfully.")
    except sqlite3.Error as e:
        print(f"Error creating user: {e}")



def delete_user(cursor):
    """
    Deletes a user from the database.
    """
    partial_username = input("Enter a partial username to search for: ").strip()
    users = search_users_by_partial_username(cursor, partial_username)

    if users:
        while True:
            try:
                choice = int(input("Select a user to delete (enter the number): ")) - 1
                if 0 <= choice < len(users):
                    break
                else:
                    print(f"Please enter a number between 1 and {len(users)}.")
            except ValueError:
                print("Invalid input. Please enter a valid number.")
        
        user_id = users[choice][0]
        
        confirm = input(f"Are you sure you want to delete user {users[choice][1]}? (yes/no): ").strip().lower()
        if confirm == 'yes':
            cursor.execute("DELETE FROM UserTable WHERE UserID = ?", (user_id,))
            print("User deleted successfully.")
        else:
            print("User deletion cancelled.")



def update_password(cursor):
    """
    Updates a user's password.
    """
    partial_username = input("Enter a partial username to search for: ").strip()
    users = search_users_by_partial_username(cursor, partial_username)

    if users:
        while True:
            try:
                choice = int(input("Select a user to update password (enter the number): ")) - 1
                if 0 <= choice < len(users):
                    break
                else:
                    print(f"Please enter a number between 1 and {len(users)}.")
            except ValueError:
                print("Invalid input. Please enter a valid number.")
        
        user_id = users[choice][0]
        
        while True:
            new_password = input("Enter the new password: ").strip()
            if (len(new_password) < 8 or not any(char.isdigit() for char in new_password) or
                    not any(char.isupper() for char in new_password) or not any(char.islower() for char in new_password) or
                    not any(char in "!@#$%^&*()-_+=<>?" for char in new_password)):
                print("Password must be at least 8 characters long and include an uppercase letter, a lowercase letter, a number, and a special character. Please try again.")
                continue
            break
        
        cursor.execute("UPDATE UserTable SET password = ?, update_date = DATE('now') WHERE UserID = ?", (new_password, user_id))
        print("Password updated successfully.")


def update_permission_level(cursor):
    """
    Updates a user's permission level.
    """
    partial_username = input("Enter a partial username to search for: ").strip()
    users = search_users_by_partial_username(cursor, partial_username)

    if users:
        while True:
            try:
                choice = int(input("Select a user to update permission level (enter the number): ")) - 1
                if 0 <= choice < len(users):
                    break
                else:
                    print(f"Please enter a number between 1 and {len(users)}.")
            except ValueError:
                print("Invalid input. Please enter a valid number.")
        
        user_id = users[choice][0]
        
        while True:
            new_permission_level = input("Enter the new permission level (1 for regular user, 2 for admin): ").strip()
            if new_permission_level not in ('1', '2'):
                print("Permission level must be either 1 or 2. Please try again.")
                continue
            break
        
        cursor.execute("UPDATE UserTable SET permission_level = ?, update_date = DATE('now') WHERE UserID = ?", (int(new_permission_level), user_id))
        print("Permission level updated successfully.")


def user_management():
    """
    Main function for user management.
    """
    db_path = os.path.join(os.getcwd(), "VegasIQ.db")
    if not os.path.exists(db_path):
        print("Database not found. Please initialize the system first.")
        return

    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        while True:
            choice = user_management_menu()
            if choice == 1:
                create_user(cursor)
            elif choice == 2:
                delete_user(cursor)
            elif choice == 3:
                partial_username = input("Enter a partial username to search for: ").strip()
                search_users_by_partial_username(cursor, partial_username)
            elif choice == 4:
                update_password(cursor)
            elif choice == 5:
                update_permission_level(cursor)
            elif choice == 6:
                print("Returning to main menu.")
                break
            else:
                print("Invalid choice. Please try again.")

            conn.commit()
            input("Press Enter to return to the user management menu...")

    except sqlite3.Error as e:
        print(f"An error occurred while managing users: {e}")

    finally:
        if 'conn' in locals() and conn:
            conn.close()








# Main program loop for VegasIQ
def main():
    """
    Main program loop for VegasIQ with integrated user authentication, dynamic menu,
    and global user tracking.
    """
    initialize_database_once()  # Ensure the database is initialized only once

    # Authenticate user before accessing the main menu
    permission_level = None
    while permission_level is None:
        clear_output(wait=True)
        display("")
        permission_level = authenticate_user()

    # User authenticated, proceed to main menu
    while True:
        clear_output(wait=True)
        display("")
        choice = display_main_menu(permission_level)
        if choice == 1:
            print(f"Option 1 selected: Las Vegas Events and Rooms Database (Placeholder). UserID: {logged_in_user_id}")
            input("Press Enter to return to the main menu...")
        elif choice == 2:
            print(f"Option 2 selected: Las Vegas Weather Data (Placeholder). UserID: {logged_in_user_id}")
            input("Press Enter to return to the main menu...")
        elif choice == 3:
            print(f"Option 3 selected: Las Vegas Airport Statistics (Placeholder). UserID: {logged_in_user_id}")
            input("Press Enter to return to the main menu...")
        elif choice == 4:
            print(f"Option 4 selected: Clark County Meeting Space Inventory (Placeholder). UserID: {logged_in_user_id}")
            input("Press Enter to return to the main menu...")
        elif choice == 5 and permission_level == 2:
            user_management()
        elif choice == 6:
            clear_output(wait=True)
            print("Thank you for using VegasIQ!")
            break
        else:
            print("Invalid choice. Please try again.")
            input("Press Enter to return to the main menu...")






main()



User Management Menu
1. Create a New User
2. Delete a User
3. Search for Users by Partial Username
4. Update Password by Partial Username Search
5. Update Permission Level by Partial Username Search
6. Return to Main Menu


''

Select an option:  5
Enter a partial username to search for:  dd



Users Found:
1. UserID: 203, Username: dd$, Name: jorge jor, Permission Level: 1


Select a user to update permission level (enter the number):  sdfsdg


ValueError: invalid literal for int() with base 10: 'sdfsdg'

Authentication successful!


Option 1 selected: Las Vegas Events and Rooms Database (Placeholder). UserID: 201
